# RQ2 — Dataset Meta-Feature Extraction

**Research Question**: What dataset properties predict which clustering method will generate the best pseudo-labels?

This notebook extracts meta-feature representations for the **94 datasets** that survived LSE computation and saves them as input tables for the meta-learner in notebook 05.

## Preprocessing pipeline (mirrors notebook 02)

Before extraction, each dataset is loaded from OpenML cache, categorical features are one-hot encoded with `dummy_na=False`, numeric NaNs are median-imputed from the training split, and all features are scaled with a train-fitted `StandardScaler`. Extractors receive the same pre-scaled `X_train`.

## Meta-feature options in the current run

| Option | Description | Dims | Output shape |
|--------|-------------|------|--------------|
| A | Hand-crafted label-free features | 20 | `(94, 29)` |
| B | Autoencoder bottleneck, fixed K=4 | 8 | `(94, 17)` |
| C | Dictionary Learning sparse codes, fixed K=4 | 8 | `(94, 17)` |
| C2 | Multi-scale order-invariant Dictionary Learning features | 27 | `(94, 36)` |
| D | Distance-based features (Ferrari & de Castro 2015) | 19 | `(94, 28)` |
| A+B | Option A plus B | 28 | `(94, 37)` |
| A+C | Option A plus C | 28 | `(94, 37)` |
| A+C2 | Option A plus C2 | 47 | `(94, 56)` |
| A+D | Option A plus D | 39 | `(94, 48)` |
| C+D | Option C plus D | 27 | `(94, 36)` |
| Rand | Random sanity baseline | 8 | `(94, 17)` |

Option A remains label-free except for `n_classes`, which is supplied by the user/deployment setting. Current validation confirms no showcase leakage, Option D has 19 features, and Option C2 has 27 features.

> Fresh-start note: delete all `mf_checkpoint_*.csv` files in `data/meta_table/` if you changed meta-feature code or preprocessing and want full recomputation.


In [1]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

ROOT    = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR = os.path.join(ROOT, 'data', 'raw')
META_DIR = os.path.join(ROOT, 'data', 'meta_table')
LSE_CSV  = os.path.join(META_DIR, 'meta_training.csv')
MANIFEST = os.path.join(META_DIR, 'dataset_manifest.csv')

sys.path.insert(0, os.path.join(ROOT, 'src'))
openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)

SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}
print('Paths OK')

Paths OK


In [2]:
lse_df = pd.read_csv(LSE_CSV)
leaked = set(lse_df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

manifest = pd.read_csv(MANIFEST)
n_cls_map = dict(zip(manifest['dataset_id'], manifest['n_classes']))

LSE_COLS    = ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']
DATASET_IDS = lse_df['dataset_id'].tolist()

print(f'Datasets to process: {len(DATASET_IDS)}')

Datasets to process: 94


In [3]:
from metafeatures import (
    extract_optA, extract_optB, extract_optC, extract_optC2, extract_optD,
    extract_optAB, extract_optAC, extract_optAC2, extract_optAD, extract_optCD,
)


def load_and_split(dataset_id):
    """Load dataset, one-hot encode categoricals, stratified 80/20 split.

    Matches notebook 02's preprocessing exactly:
      - Numeric features kept as float
      - Categorical features: pd.get_dummies(dummy_na=False) → all-zeros row for missing categoricals
      - LabelEncoder on target
    Returns raw (unscaled, un-imputed) train/test arrays — call preprocess() next.
    """
    ds = openml.datasets.get_dataset(
        dataset_id, download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )

    X_num = X.select_dtypes(include=[np.number])
    X_cat = X.select_dtypes(exclude=[np.number])
    if X_cat.shape[1] > 0:
        X_cat_enc = pd.get_dummies(X_cat, dummy_na=False).astype(float)
        X_out = pd.concat(
            [X_num.reset_index(drop=True), X_cat_enc.reset_index(drop=True)], axis=1
        )
    else:
        X_out = X_num

    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_out.values.astype(float), y_enc, test_size=0.2,
        random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def preprocess(X_tr, X_te):
    """Median imputation then StandardScaler, both fit on train only.

    Mirrors notebook 02's impute() + scale() calls.
    Returns (X_tr_scaled, X_te_scaled) as float64 arrays.
    """
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr)
    X_te = imp.transform(X_te)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)
    return X_tr_sc, X_te_sc


print('Modules loaded')

Modules loaded


## Option A — Hand-Crafted Label-Free Meta-Features (~20 features)

In [4]:
CKPT_A = os.path.join(META_DIR, 'mf_checkpoint_A.csv')

if os.path.exists(CKPT_A):
    ckpt_a = pd.read_csv(CKPT_A)
    done_a = set(ckpt_a['dataset_id'])
    rows_a = ckpt_a.to_dict('records')
    print(f'Resuming Option A — {len(done_a)} done')
else:
    done_a, rows_a = set(), []
    print('Starting Option A fresh')

total = len(DATASET_IDS)
for i, did in enumerate(DATASET_IDS):
    if did in done_a:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        X_tr_sc, _ = preprocess(X_tr, X_te)
        n_cls = n_cls_map.get(did, int(len(np.unique(y_tr))))
        feats = extract_optA(X_tr_sc, n_cls)
        feats['dataset_id'] = did
        rows_a.append(feats)
        done_a.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_a.append({'dataset_id': did})
        done_a.add(did)
    pd.DataFrame(rows_a).to_csv(CKPT_A, index=False)

optA_df = pd.DataFrame(rows_a)
print(f'\nOption A done. Shape: {optA_df.shape}')

Resuming Option A — 94 done

Option A done. Shape: (94, 21)


## Option B — Autoencoder Bottleneck (fixed K=4, → 8 dims)

In [5]:
CKPT_B = os.path.join(META_DIR, 'mf_checkpoint_B.csv')
K = 4  # fixed bottleneck size — same for all datasets

if os.path.exists(CKPT_B):
    ckpt_b = pd.read_csv(CKPT_B)
    done_b = set(ckpt_b['dataset_id'])
    rows_b = ckpt_b.to_dict('records')
    print(f'Resuming Option B — {len(done_b)} done')
else:
    done_b, rows_b = set(), []
    print('Starting Option B fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_b:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        X_tr_sc, _ = preprocess(X_tr, X_te)
        vec = extract_optB(X_tr_sc, K=K)
        rec = {'dataset_id': did}
        for j, v in enumerate(vec[:K]):
            rec[f'ae_mean_{j}'] = float(v)
        for j, v in enumerate(vec[K:]):
            rec[f'ae_var_{j}'] = float(v)
        rows_b.append(rec)
        done_b.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_b.append({'dataset_id': did})
        done_b.add(did)
    pd.DataFrame(rows_b).to_csv(CKPT_B, index=False)

optB_df = pd.DataFrame(rows_b)
print(f'\nOption B done. Shape: {optB_df.shape}')

Resuming Option B — 94 done

Option B done. Shape: (94, 9)


## Option C — Dictionary Learning Sparse Codes (fixed K=4, → 8 dims)

In [6]:
CKPT_C = os.path.join(META_DIR, 'mf_checkpoint_C.csv')

if os.path.exists(CKPT_C):
    ckpt_c = pd.read_csv(CKPT_C)
    done_c = set(ckpt_c['dataset_id'])
    rows_c = ckpt_c.to_dict('records')
    print(f'Resuming Option C — {len(done_c)} done')
else:
    done_c, rows_c = set(), []
    print('Starting Option C fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_c:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        X_tr_sc, _ = preprocess(X_tr, X_te)
        vec = extract_optC(X_tr_sc, K=K)
        rec = {'dataset_id': did}
        for j, v in enumerate(vec[:K]):
            rec[f'dl_mean_{j}'] = float(v)
        for j, v in enumerate(vec[K:]):
            rec[f'dl_var_{j}'] = float(v)
        rows_c.append(rec)
        done_c.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_c.append({'dataset_id': did})
        done_c.add(did)
    pd.DataFrame(rows_c).to_csv(CKPT_C, index=False)

optC_df = pd.DataFrame(rows_c)
print(f'\nOption C done. Shape: {optC_df.shape}')

Resuming Option C — 94 done

Option C done. Shape: (94, 9)


## Option C2 — Multi-Scale Order-Invariant DL Features (K=4,8,16 → 27 dims)

Experimental improvement over Option C. The current run completed all 94 datasets and produced `(94, 28)` before targets are merged: one dataset ID column plus 27 meta-features.

C2 addresses three limitations of fixed-K dictionary features:
1. **K=4 too small** — features are extracted at K=4, K=8, and K=16; K is skipped if K > n_features
2. **Atom-order dependence** — aggregations are global/distributional instead of per-atom
3. **Missing reconstruction signal** — mean and CV of per-row reconstruction error are included

The sanity diagnostic found 92 complete rows out of 94 and median pairwise cosine distance 0.2092, indicating the features are dataset-discriminative.


In [7]:
CKPT_C2 = os.path.join(META_DIR, 'mf_checkpoint_C2.csv')

if os.path.exists(CKPT_C2):
    ckpt_c2 = pd.read_csv(CKPT_C2)
    done_c2 = set(ckpt_c2['dataset_id'])
    rows_c2 = ckpt_c2.to_dict('records')
    print(f'Resuming Option C2 — {len(done_c2)} done')
else:
    done_c2, rows_c2 = set(), []
    print('Starting Option C2 fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_c2:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        X_tr_sc, _ = preprocess(X_tr, X_te)
        feats = extract_optC2(X_tr_sc)
        feats['dataset_id'] = did
        rows_c2.append(feats)
        done_c2.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_c2.append({'dataset_id': did})
        done_c2.add(did)
    pd.DataFrame(rows_c2).to_csv(CKPT_C2, index=False)

optC2_df = pd.DataFrame(rows_c2)
c2_feat_cols = [c for c in optC2_df.columns if c != 'dataset_id']
print(f'\nOption C2 done. Shape: {optC2_df.shape}  ({len(c2_feat_cols)} features, expected 27)')

Starting Option C2 fresh
[  1/94]  did=44507  ok  (59.7s)
[  2/94]  did=44227  ok  (26.0s)
[  3/94]  did=44624  ok  (182.5s)
[  4/94]  did=41168  ok  (231.9s)
[  5/94]  did=1465  ok  (0.8s)
[  6/94]  did=830  ok  (5.3s)
[  7/94]  did=44744  ok  (10.9s)
[  8/94]  did=44701  ok  (135.9s)
[  9/94]  did=46553  ok  (158.7s)
[ 10/94]  did=694  ok  (1.9s)
[ 11/94]  did=44232  ok  (30.1s)
[ 12/94]  did=42544  ok  (1.2s)
[ 13/94]  did=46745  ok  (43.7s)
[ 14/94]  did=1044  ok  (202.3s)
[ 15/94]  did=46560  ok  (148.8s)
[ 16/94]  did=46331  ok  (6.5s)
[ 17/94]  did=40994  ok  (28.3s)
[ 18/94]  did=60  ok  (309.5s)
[ 19/94]  did=44463  ok  (42.5s)
[ 20/94]  did=1559  ok  (1.8s)
[ 21/94]  did=46840  ok  (58.5s)
[ 22/94]  did=1519  ok  (2.2s)
[ 23/94]  did=756  ok  (4.0s)
[ 24/94]  did=1477  ok  (266.6s)
[ 25/94]  did=1552  ok  (46.1s)
[ 26/94]  did=44683  ok  (153.7s)
[ 27/94]  did=44073  ok  (49.8s)
[ 28/94]  did=44605  ok  (8.9s)
[ 29/94]  did=41764  ok  (30.4s)
[ 30/94]  did=41671  ok  (61.8s)


In [8]:
from scipy.spatial.distance import pdist as _pdist

# Sanity diagnostic: are C2 features dataset-discriminative?
df_c2_valid = optC2_df.dropna(subset=c2_feat_cols)
C2_mat = df_c2_valid[c2_feat_cols].values.astype(float)

print(f'=== Option C2 Sanity Diagnostic ===')
print(f'Complete rows: {len(df_c2_valid)} / {len(optC2_df)}')
print(f'Feature dimension: {len(c2_feat_cols)} (expected 27)')

# Sparsity values should be in [0, 1]
sparsity_cols = [c for c in c2_feat_cols if '_sparsity' in c]
print(f'\nSparsity features (should be in [0, 1]):')
for col in sparsity_cols:
    v = df_c2_valid[col]
    print(f'  {col}: min={v.min():.3f}  mean={v.mean():.3f}  max={v.max():.3f}')

# Reconstruction error should be positive
recon_cols = [c for c in c2_feat_cols if '_recon_mean' in c]
print(f'\nReconstruction error (should be > 0):')
for col in recon_cols:
    v = df_c2_valid[col]
    print(f'  {col}: min={v.min():.3f}  mean={v.mean():.3f}  max={v.max():.3f}')

# Pairwise cosine distances — low median means features aren't discriminating datasets
cos_dists = _pdist(C2_mat, metric='cosine')
print(f'\nPairwise cosine distance distribution ({len(cos_dists):,} pairs):')
for pct in (0, 10, 25, 50, 75, 90, 100):
    print(f'  p{pct:3d}: {np.percentile(cos_dists, pct):.4f}')

median_cos = float(np.median(cos_dists))
if median_cos < 0.05:
    print(f'\nWARNING: median cosine distance = {median_cos:.4f} < 0.05 — features may not be dataset-discriminative')
else:
    print(f'\nOK: median cosine distance = {median_cos:.4f} — features look dataset-discriminative')

=== Option C2 Sanity Diagnostic ===
Complete rows: 92 / 94
Feature dimension: 27 (expected 27)

Sparsity features (should be in [0, 1]):
  dl2_k4_sparsity: min=0.132  mean=0.446  max=0.992
  dl2_k8_sparsity: min=0.222  mean=0.573  max=0.985
  dl2_k16_sparsity: min=0.000  mean=0.554  max=0.985

Reconstruction error (should be > 0):
  dl2_k4_recon_mean: min=0.603  mean=5.952  max=38.232
  dl2_k8_recon_mean: min=0.557  mean=5.412  max=38.129
  dl2_k16_recon_mean: min=0.000  mean=4.607  max=37.923

Pairwise cosine distance distribution (4,186 pairs):
  p  0: 0.0000
  p 10: 0.0335
  p 25: 0.0823
  p 50: 0.2092
  p 75: 0.3304
  p 90: 0.4381
  p100: 0.6657

OK: median cosine distance = 0.2092 — features look dataset-discriminative


## Option D — Distance-Based Meta-Features (Ferrari & de Castro 2015, 19 dims)

In [9]:
CKPT_D = os.path.join(META_DIR, 'mf_checkpoint_D.csv')

if os.path.exists(CKPT_D):
    ckpt_d = pd.read_csv(CKPT_D)
    done_d = set(ckpt_d['dataset_id'])
    rows_d = ckpt_d.to_dict('records')
    print(f'Resuming Option D — {len(done_d)} done')
else:
    done_d, rows_d = set(), []
    print('Starting Option D fresh')

for i, did in enumerate(DATASET_IDS):
    if did in done_d:
        continue
    t0 = time.time()
    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)
        X_tr_sc, _ = preprocess(X_tr, X_te)
        vec = extract_optD(X_tr_sc)
        assert len(vec) == 19, f'Option D vector length {len(vec)} != 19'
        rec = {'dataset_id': did}
        for j, v in enumerate(vec):
            rec[f'md{j+1}'] = float(v)
        rows_d.append(rec)
        done_d.add(did)
        print(f'[{i+1:3d}/{total}]  did={did}  ok  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'[{i+1:3d}/{total}]  did={did}  FAIL: {e}')
        rows_d.append({'dataset_id': did})
        done_d.add(did)
    pd.DataFrame(rows_d).to_csv(CKPT_D, index=False)

optD_df = pd.DataFrame(rows_d)
print(f'\nOption D done. Shape: {optD_df.shape}')

# Verify D vector length on first successful row
d_cols = [c for c in optD_df.columns if c != 'dataset_id' and not optD_df[c].isna().all()]
print(f'Option D feature count: {len(d_cols)} (expected 19)')

Resuming Option D — 94 done

Option D done. Shape: (94, 20)
Option D feature count: 19 (expected 19)


## Concatenated Variants (A+B, A+C, A+C2, A+D, C+D)

Built by merging individual option dataframes; no separate extraction is needed. These variants test whether learned, multi-scale dictionary, or distance-based representations add signal beyond the hand-crafted Option A features.


In [10]:
def _mf_cols(df):
    """Return all columns except dataset_id from a meta-feature dataframe."""
    return [c for c in df.columns if c != 'dataset_id']


def _concat_opts(df_left, df_right):
    right_extra = ['dataset_id'] + [c for c in _mf_cols(df_right) if c not in _mf_cols(df_left)]
    return df_left.merge(df_right[right_extra], on='dataset_id', how='inner')


optAB_df  = _concat_opts(optA_df,  optB_df)
optAC_df  = _concat_opts(optA_df,  optC_df)
optAC2_df = _concat_opts(optA_df,  optC2_df)
optAD_df  = _concat_opts(optA_df,  optD_df)
optCD_df  = _concat_opts(optC_df,  optD_df)

for suffix, df in [('AB', optAB_df), ('AC', optAC_df), ('AC2', optAC2_df),
                    ('AD', optAD_df), ('CD', optCD_df)]:
    print(f'Option {suffix}: {df.shape}  ({len(_mf_cols(df))} meta-features)')

Option AB: (94, 29)  (28 meta-features)
Option AC: (94, 29)  (28 meta-features)
Option AC2: (94, 48)  (47 meta-features)
Option AD: (94, 40)  (39 meta-features)
Option CD: (94, 28)  (27 meta-features)


## Sanity Baseline — Random Meta-Features

Random 8-dimensional features (matching Options B and C dimensionality).
If Options B or C cannot beat random in the meta-learner, they are not learning
useful representations — the result would be noise, not a contribution.

In [11]:
rng = np.random.default_rng(SEED)
random_rows = []
for did in DATASET_IDS:
    rec = {'dataset_id': did}
    for j in range(2 * K):
        rec[f'rand_{j}'] = float(rng.normal())
    random_rows.append(rec)

optRand_df = pd.DataFrame(random_rows)
print(f'Random features shape: {optRand_df.shape}')

Random features shape: (94, 9)


## Merge & Save All Tables

The current run writes 11 option tables plus the updated main `meta_training.csv`. The main table now has shape `(94, 49)` after appending Option A columns to the LSE targets.


In [12]:
TARGET_COLS = ['dataset_id'] + LSE_COLS + ['best_method', 'gt_accuracy']
targets = lse_df[TARGET_COLS]

for letter, mf_df in [
    ('A',    optA_df),
    ('B',    optB_df),
    ('C',    optC_df),
    ('C2',   optC2_df),
    ('D',    optD_df),
    ('AB',   optAB_df),
    ('AC',   optAC_df),
    ('AC2',  optAC2_df),
    ('AD',   optAD_df),
    ('CD',   optCD_df),
    ('Rand', optRand_df),
]:
    merged = targets.merge(mf_df, on='dataset_id', how='inner')
    out_path = os.path.join(META_DIR, f'meta_training_opt{letter}.csv')
    merged.to_csv(out_path, index=False)
    leaked = set(merged['dataset_id']) & SHOWCASE_IDS
    assert len(leaked) == 0, f'Showcase leak in opt{letter}: {leaked}'
    n_mf = len([c for c in merged.columns if c not in TARGET_COLS])
    print(f'Option {letter:4s} → {out_path.split(os.sep)[-1]}  shape={merged.shape}  meta-features={n_mf}')

# Update main meta_training.csv with Option A features
main_df = lse_df.merge(optA_df, on='dataset_id', how='left')
main_df.to_csv(LSE_CSV, index=False)
print(f'\nMain table updated → {LSE_CSV}  shape={main_df.shape}')

Option A    → meta_training_optA.csv  shape=(94, 29)  meta-features=20
Option B    → meta_training_optB.csv  shape=(94, 17)  meta-features=8
Option C    → meta_training_optC.csv  shape=(94, 17)  meta-features=8
Option C2   → meta_training_optC2.csv  shape=(94, 36)  meta-features=27
Option D    → meta_training_optD.csv  shape=(94, 28)  meta-features=19
Option AB   → meta_training_optAB.csv  shape=(94, 37)  meta-features=28
Option AC   → meta_training_optAC.csv  shape=(94, 37)  meta-features=28
Option AC2  → meta_training_optAC2.csv  shape=(94, 56)  meta-features=47
Option AD   → meta_training_optAD.csv  shape=(94, 48)  meta-features=39
Option CD   → meta_training_optCD.csv  shape=(94, 36)  meta-features=27
Option Rand → meta_training_optRand.csv  shape=(94, 17)  meta-features=8

Main table updated → c:\MLResearch\data\meta_table\meta_training.csv  shape=(94, 49)


In [13]:
df_a = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
mf_cols = [c for c in df_a.columns if c not in TARGET_COLS]

print(f'=== Option A: {len(mf_cols)} meta-features ===')
print('Features:', mf_cols)

# Confirm no label-derived columns survived
LABEL_FEATURES = {
    'silhouette_true', 'davies_bouldin_true', 'knn1_accuracy',
    'decision_stump_accuracy', 'inter_intra_ratio', 'class_entropy', 'imbalance_ratio',
}
surviving = LABEL_FEATURES & set(mf_cols)
assert len(surviving) == 0, f'Label-dependent features still present: {surviving}'
print('Label-free check: PASSED — no label-derived features in Option A')

nan_counts = df_a[mf_cols].isna().sum()
if nan_counts.any():
    print('\nNaN counts (Option A):')
    print(nan_counts[nan_counts > 0])
else:
    print('\nNo NaN values in Option A.')

# Verify Option D vector length
df_d = pd.read_csv(os.path.join(META_DIR, 'meta_training_optD.csv'))
d_feat_cols = [c for c in df_d.columns if c not in TARGET_COLS]
assert len(d_feat_cols) == 19, f'Option D has {len(d_feat_cols)} features, expected 19'
print(f'\nOption D vector length: {len(d_feat_cols)} ✓')

# Verify Option C2 vector length
df_c2 = pd.read_csv(os.path.join(META_DIR, 'meta_training_optC2.csv'))
c2_feat_cols_val = [c for c in df_c2.columns if c not in TARGET_COLS]
assert len(c2_feat_cols_val) == 27, f'Option C2 has {len(c2_feat_cols_val)} features, expected 27'
print(f'Option C2 vector length: {len(c2_feat_cols_val)} ✓')

# Verify all 11 output files exist and have no showcase leaks
expected_files = [f'meta_training_opt{s}.csv'
                  for s in ['A', 'B', 'C', 'C2', 'D', 'AB', 'AC', 'AC2', 'AD', 'CD', 'Rand']]
for fname in expected_files:
    path = os.path.join(META_DIR, fname)
    assert os.path.exists(path), f'Missing: {fname}'
    df_tmp = pd.read_csv(path)
    leaked = set(df_tmp['dataset_id']) & SHOWCASE_IDS
    assert len(leaked) == 0, f'Showcase leak in {fname}: {leaked}'
    print(f'  {fname:40s}  shape={df_tmp.shape}')

print('\nAll sanity checks passed.')
print('Ready for 05_meta_learner.ipynb')

=== Option A: 20 meta-features ===
Features: ['n_instances', 'n_features', 'n_classes', 'skewness_mean', 'kurtosis_mean', 'mean_abs_pearson', 'hopkins', 'intrinsic_dim_ratio', 'pca_var_pc1', 'pca_top3_var', 'pca_entropy', 'pairwise_dist_mean', 'pairwise_dist_cv', 'pairwise_dist_p90', 'knn5_dist_mean', 'knn5_dist_cv', 'knn5_dist_p90', 'feature_sparsity', 'high_corr_frac', 'corr_dispersion']
Label-free check: PASSED — no label-derived features in Option A

NaN counts (Option A):
skewness_mean       22
kurtosis_mean       22
mean_abs_pearson    22
hopkins              9
dtype: int64

Option D vector length: 19 ✓
Option C2 vector length: 27 ✓
  meta_training_optA.csv                    shape=(94, 29)
  meta_training_optB.csv                    shape=(94, 17)
  meta_training_optC.csv                    shape=(94, 17)
  meta_training_optC2.csv                   shape=(94, 36)
  meta_training_optD.csv                    shape=(94, 28)
  meta_training_optAB.csv                   shape=(94, 37)